診斷腳本（只診斷、不清理）
=================================================================
判斷 austin_animal_intakes.csv / austin_animal_outcomes.csv 這兩份「更新資料」
能否、以及如何被清理成模型的 out-of-time test set。

模型背景：
- 預測 is_long_stay（單次住院 > 30 天），只針對 Dog / Cat。
- 訓練母體：進入收容所、等待 outcome（認養/領回/轉出/安樂死）的貓狗。
- 訓練 long-stay 率參考：Dog ~0.16、Cat ~0.25。
- 訓練特徵（7 個）：
    [intake_reason, primary_breed, intake_health_condition, sex, age_days, is_sn, is_mix]

本腳本不做任何過濾/排除，只印出證據。
用法：python3 diagnose_new_system_data.py

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# 新舊資料的時間分界（舊訓練資料最後一筆 outcome 的日期）
NEW_DATA_BOUNDARY = pd.Timestamp("2025-05-05")
LONG_STAY_THRESHOLD = 30  # days

INTAKES_PATH = "../../data/new_system_data/austin_animal_intakes.csv"
OUTCOMES_PATH = "../../data/new_system_data/austin_animal_outcomes.csv"

# 訓練用的 7 個特徵（用於 schema 對映缺口檢查）
TRAIN_FEATURES = [
    "intake_reason",
    "primary_breed",
    "intake_health_condition",
    "sex",
    "age_days",
    "is_sn",
    "is_mix",
]


def banner(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def subhdr(title):
    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)

In [2]:
# =====================================================================
# 載入
# =====================================================================
intakes = pd.read_csv(INTAKES_PATH)
outcomes = pd.read_csv(OUTCOMES_PATH)

# 日期欄一律用 errors='coerce'
intakes["source_date"] = pd.to_datetime(intakes["source_date"], errors="coerce")
outcomes["outcome_date"] = pd.to_datetime(outcomes["outcome_date"], errors="coerce")
outcomes["intake_date"] = pd.to_datetime(outcomes["intake_date"], errors="coerce")

In [3]:
# =====================================================================
# 1. 基本資訊
# =====================================================================
banner("[1] 基本資訊：shape / 欄位 / 日期範圍 / 新舊銜接")

subhdr("[1a] shape 與欄位列表")
print(f"intakes  shape : {intakes.shape}")
print(f"intakes  columns ({len(intakes.columns)}): {list(intakes.columns)}")
print()
print(f"outcomes shape : {outcomes.shape}")
print(f"outcomes columns ({len(outcomes.columns)}): {list(outcomes.columns)}")

subhdr("[1b] 日期範圍")
print(f"intakes.source_date   : {intakes['source_date'].min()}  ->  {intakes['source_date'].max()}"
      f"   (NaT: {intakes['source_date'].isna().sum()})")
print(f"outcomes.outcome_date : {outcomes['outcome_date'].min()}  ->  {outcomes['outcome_date'].max()}"
      f"   (NaT: {outcomes['outcome_date'].isna().sum()})")
print(f"outcomes.intake_date  : {outcomes['intake_date'].min()}  ->  {outcomes['intake_date'].max()}"
      f"   (NaT: {outcomes['intake_date'].isna().sum()})")

subhdr(f"[1c] 新舊銜接：outcome_date 是否都晚於 {NEW_DATA_BOUNDARY.date()}")
valid_out = outcomes["outcome_date"].notna().sum()
after = (outcomes["outcome_date"] >= NEW_DATA_BOUNDARY).sum()
before = (outcomes["outcome_date"] < NEW_DATA_BOUNDARY).sum()
print(f"有效 outcome_date 筆數          : {valid_out}")
print(f"outcome_date >= {NEW_DATA_BOUNDARY.date()}      : {after}  ({after / valid_out:.1%})")
print(f"outcome_date <  {NEW_DATA_BOUNDARY.date()} (重疊舊資料) : {before}  ({before / valid_out:.1%})")
if before > 0:
    print(">> 警告：有 outcome 落在舊訓練資料時間範圍內，做 out-of-time test set 時需考慮是否剔除以避免時間洩漏。")


[1] 基本資訊：shape / 欄位 / 日期範圍 / 新舊銜接

----------------------------------------------------------------------
[1a] shape 與欄位列表
----------------------------------------------------------------------
intakes  shape : (14132, 15)
intakes  columns (15): ['id', 'source_date', 'animal_id', 'type', 'source_name', 'name_at_intake', 'ispreviouslyspayedneutered', 'sex', 'primary_breed', 'primary_color', 'secondary_color', 'intake_health_condition', 'date_of_birth', 'found_address', 'timestamp']

outcomes shape : (13068, 16)
outcomes columns (16): ['id', 'outcome_date', 'animal_id', 'name', 'outcome_status', 'euthanasia_reason', 'type', 'sex', 'spayed_neutered', 'primary_breed', 'primary_color', 'secondary_color', 'date_of_birth', 'intake_date', 'days_in_shelter', 'timestamp']

----------------------------------------------------------------------
[1b] 日期範圍
----------------------------------------------------------------------
intakes.source_date   : 2025-05-04 20:45:00  ->  2026-06-18 17:45:00   (N

In [4]:
# =====================================================================
# 用 merge_asof 把每筆 outcome 配到「緊鄰其前、同一隻動物」的 intake
# =====================================================================
banner("[merge_asof] 將 outcomes 與 intakes 依 animal_id + 時間對齊配對")

# merge_asof 要求兩邊都先依「on」鍵全域排序
intakes_sorted = intakes.dropna(subset=["source_date"]).sort_values("source_date")
outcomes_sorted = outcomes.dropna(subset=["outcome_date"]).sort_values("outcome_date")

print(f"參與配對的 intakes  （source_date 非 NaT）: {len(intakes_sorted)} / {len(intakes)}")
print(f"參與配對的 outcomes （outcome_date 非 NaT）: {len(outcomes_sorted)} / {len(outcomes)}")

merged = pd.merge_asof(
    outcomes_sorted,
    intakes_sorted[["animal_id", "source_date", "source_name", "type"]].rename(
        columns={"type": "intake_type"}
    ),
    left_on="outcome_date",
    right_on="source_date",
    by="animal_id",
    direction="backward",
)

n_matched = merged["source_date"].notna().sum()
n_unmatched = merged["source_date"].isna().sum()
print(f"配到 intake 的 outcome 筆數 : {n_matched}  ({n_matched / len(merged):.1%})")
print(f"配不到 intake 的 outcome 筆數: {n_unmatched}  ({n_unmatched / len(merged):.1%})  (source_date 為 NaT)")


[merge_asof] 將 outcomes 與 intakes 依 animal_id + 時間對齊配對
參與配對的 intakes  （source_date 非 NaT）: 14132 / 14132
參與配對的 outcomes （outcome_date 非 NaT）: 13068 / 13068
配到 intake 的 outcome 筆數 : 12128  (92.8%)
配不到 intake 的 outcome 筆數: 940  (7.2%)  (source_date 為 NaT)


In [5]:
# =====================================================================
# 2. source_name × outcome_status 交叉分析（核心）
# =====================================================================
banner("[2] source_name x outcome_status 交叉分析")
print("目的：看哪些 source_name 大量走 'Return to Habitat'/'Released' 等放回流程（應排除），")
print("      哪些走 'Adopted'/'Transfer Out'/'Reclaimed' 等正常收容流程（應保留）。")
print(f"（僅含配到 intake 的 {n_matched} 筆 outcome）")

matched = merged[merged["source_date"].notna()].copy()

subhdr("[2a] 各 source_name 的總筆數")
print(matched["source_name"].value_counts(dropna=False).to_string())

subhdr("[2b] 每個 source_name 內的 outcome_status 分布（count 與 % within source_name）")
for sname, grp in matched.groupby("source_name", dropna=False):
    cnt = grp["outcome_status"].value_counts(dropna=False)
    pct = grp["outcome_status"].value_counts(dropna=False, normalize=True)
    tbl = pd.DataFrame({"count": cnt, "pct": (pct * 100).round(1)})
    print(f"\n### source_name = {sname!r}   (n={len(grp)})")
    print(tbl.to_string())


[2] source_name x outcome_status 交叉分析
目的：看哪些 source_name 大量走 'Return to Habitat'/'Released' 等放回流程（應排除），
      哪些走 'Adopted'/'Transfer Out'/'Reclaimed' 等正常收容流程（應保留）。
（僅含配到 intake 的 12128 筆 outcome）

----------------------------------------------------------------------
[2a] 各 source_name 的總筆數
----------------------------------------------------------------------
source_name
Stray                                3852
ACO - Impound                        3374
Community Animal - Program           2754
Owner Surrender                      1129
Returns                               294
Ambulance                             268
Agency                                228
Abandoned                              74
ACO - Owner Surrender                  54
Shelter Offspring                      50
Foster Offspring                       13
Ambulance - Owner Surrender             8
Community Animal - Field Services       8
Humane Officer Surrendered              7
Wildlife                           

In [6]:
# =====================================================================
# 3. 重算「單次住院」天數，對照被灌水的 days_in_shelter
# =====================================================================
banner("[3] 重算單次住院天數 los_recomputed vs 原檔 days_in_shelter")
print("los_recomputed = (outcome_date - 配到的 intake.source_date).days")
print("原檔 days_in_shelter = outcome_date - intake_date（intake_date 可能跨多年，被灌水，不可信）")

merged["los_recomputed"] = (merged["outcome_date"] - merged["source_date"]).dt.days

subhdr("[3a] describe() 並排：days_in_shelter（原檔）vs los_recomputed")
cmp_desc = pd.DataFrame({
    "days_in_shelter(原檔)": pd.to_numeric(merged["days_in_shelter"], errors="coerce").describe(),
    "los_recomputed":        merged["los_recomputed"].describe(),
})
print(cmp_desc.to_string())

subhdr("[3b] 兩者不一致比例 / 配不到 intake 筆數")
both = merged.dropna(subset=["los_recomputed"]).copy()
both["days_in_shelter_num"] = pd.to_numeric(both["days_in_shelter"], errors="coerce")
both_valid = both.dropna(subset=["days_in_shelter_num"])
mismatch = (both_valid["days_in_shelter_num"] != both_valid["los_recomputed"]).sum()
print(f"可比較筆數（兩欄皆有值）        : {len(both_valid)}")
print(f"days_in_shelter != los_recomputed: {mismatch}  ({mismatch / len(both_valid):.1%})")
print(f"配不到 intake（los_recomputed NaN）: {merged['los_recomputed'].isna().sum()}")
neg = (merged["los_recomputed"] < 0).sum()
print(f"los_recomputed < 0（資料異常）   : {neg}")

subhdr("[3c] 用 los_recomputed > 30 重算 is_long_stay，按 type 分組（對照 Dog~0.16 / Cat~0.25）")


def normalize_type(s):
    """Kitten->Cat, Puppy->Dog；其餘原樣。"""
    return s.replace({"Kitten": "Cat", "Puppy": "Dog"})


valid_los = merged.dropna(subset=["los_recomputed"]).copy()
valid_los = valid_los[valid_los["los_recomputed"] >= 0]
valid_los["type_norm"] = normalize_type(valid_los["type"])
valid_los["is_long_stay_recomp"] = valid_los["los_recomputed"] > LONG_STAY_THRESHOLD

ls_by_type = valid_los.groupby("type_norm")["is_long_stay_recomp"].agg(["mean", "count"])
ls_by_type.columns = ["long_stay_rate", "n"]
print("（全部 type，type_norm 已把 Kitten/Puppy 併回 Cat/Dog）")
print(ls_by_type.sort_values("n", ascending=False).to_string())

print("\n>> 只看 Dog / Cat（模型母體）：")
dogcat = ls_by_type.reindex(["Dog", "Cat"])
print(dogcat.to_string())

# 對照：若改用原檔 days_in_shelter 算出的 long-stay 率（示範灌水後的不合理值）
valid_los["days_num"] = pd.to_numeric(valid_los["days_in_shelter"], errors="coerce")
orig = valid_los.dropna(subset=["days_num"]).copy()
orig["is_long_orig"] = orig["days_num"] > LONG_STAY_THRESHOLD
orig_rate = orig.groupby("type_norm")["is_long_orig"].mean().reindex(["Dog", "Cat"])
print("\n>> 對照（用原檔 days_in_shelter 算的 long-stay 率，應明顯偏高/不合理）：")
print(orig_rate.to_string())


[3] 重算單次住院天數 los_recomputed vs 原檔 days_in_shelter
los_recomputed = (outcome_date - 配到的 intake.source_date).days
原檔 days_in_shelter = outcome_date - intake_date（intake_date 可能跨多年，被灌水，不可信）

----------------------------------------------------------------------
[3a] describe() 並排：days_in_shelter（原檔）vs los_recomputed
----------------------------------------------------------------------
       days_in_shelter(原檔)  los_recomputed
count         13068.000000    12128.000000
mean            239.874732       22.474027
std             141.662122       37.267481
min               1.000000        0.000000
25%             112.000000        3.000000
50%             254.000000        8.000000
75%             357.000000       27.000000
max            1935.000000      376.000000

----------------------------------------------------------------------
[3b] 兩者不一致比例 / 配不到 intake 筆數
----------------------------------------------------------------------
可比較筆數（兩欄皆有值）        : 12128
days_in_shelter != los_rec

In [7]:
# =====================================================================
# 4. multi-visit 量化
# =====================================================================
banner("[4] multi-visit 量化")

intake_counts = intakes["animal_id"].value_counts()
outcome_counts = outcomes["animal_id"].value_counts()

subhdr("[4a] 每個 animal_id 在 intakes / outcomes 的出現次數分布")
print("intakes  每隻動物 intake 次數分布：")
print(intake_counts.value_counts().sort_index().rename("animals").to_string())
print("\noutcomes 每隻動物 outcome 次數分布：")
print(outcome_counts.value_counts().sort_index().rename("animals").to_string())

subhdr("[4b] multi-visit 動物數與佔比")
multi_intake = (intake_counts > 1).sum()
multi_outcome = (outcome_counts > 1).sum()
print(f"intakes  唯一 animal_id 數 : {intake_counts.size}")
print(f"  >1 次 intake 的動物      : {multi_intake}  ({multi_intake / intake_counts.size:.1%})")
print(f"outcomes 唯一 animal_id 數 : {outcome_counts.size}")
print(f"  >1 次 outcome 的動物     : {multi_outcome}  ({multi_outcome / outcome_counts.size:.1%})")

subhdr("[4c] 在 multi-visit 動物上，days_in_shelter（原檔）vs los_recomputed 的灌水程度")
multi_ids = set(outcome_counts[outcome_counts > 1].index) | set(intake_counts[intake_counts > 1].index)
mv = merged[merged["animal_id"].isin(multi_ids)].dropna(subset=["los_recomputed"]).copy()
mv["days_num"] = pd.to_numeric(mv["days_in_shelter"], errors="coerce")
mv = mv.dropna(subset=["days_num"])
mv["inflation"] = mv["days_num"] - mv["los_recomputed"]
print(f"multi-visit 且可比較的 outcome 筆數 : {len(mv)}")
if len(mv) > 0:
    print(f"days_in_shelter 中位數 : {mv['days_num'].median():.1f}")
    print(f"los_recomputed 中位數  : {mv['los_recomputed'].median():.1f}")
    print(f"差異(灌水) 中位數      : {mv['inflation'].median():.1f} 天")
    print(f"差異(灌水) 平均        : {mv['inflation'].mean():.1f} 天")
    print("\n差異(灌水) describe()：")
    print(mv["inflation"].describe().to_string())


[4] multi-visit 量化

----------------------------------------------------------------------
[4a] 每個 animal_id 在 intakes / outcomes 的出現次數分布
----------------------------------------------------------------------
intakes  每隻動物 intake 次數分布：
count
1    13176
2      408
3       35
4        5
5        3

outcomes 每隻動物 outcome 次數分布：
count
1    12231
2      360
3       25
4        8
5        2

----------------------------------------------------------------------
[4b] multi-visit 動物數與佔比
----------------------------------------------------------------------
intakes  唯一 animal_id 數 : 13627
  >1 次 intake 的動物      : 451  (3.3%)
outcomes 唯一 animal_id 數 : 12626
  >1 次 outcome 的動物     : 395  (3.1%)

----------------------------------------------------------------------
[4c] 在 multi-visit 動物上，days_in_shelter（原檔）vs los_recomputed 的灌水程度
----------------------------------------------------------------------
multi-visit 且可比較的 outcome 筆數 : 890
days_in_shelter 中位數 : 232.5
los_recomputed 中位數  : 12.0
差異(灌水) 中

In [8]:
# =====================================================================
# 5. schema 對映缺口清單
# =====================================================================
banner("[5] schema 對映缺口清單")

subhdr("[5a] 新資料 type 欄類別與數量（Kitten/Puppy 需併回 Cat/Dog）")
print("intakes.type：")
print(intakes["type"].value_counts(dropna=False).to_string())
print("\nintakes.type（Kitten->Cat, Puppy->Dog 併回後）：")
print(normalize_type(intakes["type"]).value_counts(dropna=False).to_string())
print("\noutcomes.type（Kitten->Cat, Puppy->Dog 併回後）：")
print(normalize_type(outcomes["type"]).value_counts(dropna=False).to_string())

subhdr("[5b] 7 個訓練特徵 → 新資料對映狀態")
# (feature, 新資料來源檔.欄位, 型態/處理註記)
mapping = {
    "intake_reason": (
        "intakes.source_name (21 類)",
        "型態一致（字串/類別）。疑似對應舊 intake_reason，但類別集合不同，"
        "需重新對映；且含 'Community Animal - Program' 等放回流程類別需先用 [2] 判斷是否排除。",
    ),
    "primary_breed": (
        "intakes.primary_breed / outcomes.primary_breed",
        "已預先拆好（另有 secondary_color 等），不需再拆 breed 字串。型態一致（字串）。",
    ),
    "intake_health_condition": (
        "intakes.intake_health_condition",
        "欄名一致，型態為字串。需檢查類別值是否與舊資料對齊（見下方值分布）。",
    ),
    "sex": (
        "intakes.sex / outcomes.sex",
        "型態一致（字串）。注意 intakes.sex 多為 'Unknown'（見值分布），"
        "outcomes.sex 為 Male/Female/Unknown；新資料未把絕育狀態編進 sex（舊資料可能是 'Neutered Male' 形式）。",
    ),
    "age_days": (
        "需由 date_of_birth 計算（兩檔皆有 date_of_birth）",
        "無現成 age_days 欄；需 (參考日期 - date_of_birth).days 推算。"
        "參考日：intake 端用 source_date、outcome 端用 outcome_date。型態：需自行衍生。",
    ),
    "is_sn": (
        "intakes.ispreviouslyspayedneutered (bool) / outcomes.spayed_neutered (字串)",
        "型態不一致！intakes 是布林 True/False；outcomes 是字串 Yes/No/Unknown。"
        "舊訓練資料 is_sn 來源型態需確認，做 test set 時要統一成同一編碼。",
    ),
    "is_mix": (
        "★ 完全缺失（無對應欄）",
        "新資料只有 primary_breed / secondary_color，無 'is_mix' 或 'Mix' 標記欄位，"
        "也無 breed 字串可解析出 'Mix'。無法直接還原此特徵 —— 需決定填補策略或視為缺失。",
    ),
}

cols_intake = set(intakes.columns)
cols_outcome = set(outcomes.columns)
rows = []
for feat in TRAIN_FEATURES:
    src, note = mapping[feat]
    rows.append({"feature": feat, "新資料來源 / 欄位": src, "型態/處理註記": note})
gap = pd.DataFrame(rows)
print(gap.to_string(index=False))

print("\n>> 明確標出『完全缺失』的特徵：")
print("   - is_mix：新資料無任何對應欄位，且無 breed 字串可解析，無法直接重建。")

subhdr("[5c] 相關欄位的實際值分布（供判斷型態/類別對齊）")
for col, df, lbl in [
    ("intake_health_condition", intakes, "intakes"),
    ("sex", intakes, "intakes"),
    ("sex", outcomes, "outcomes"),
    ("ispreviouslyspayedneutered", intakes, "intakes"),
    ("spayed_neutered", outcomes, "outcomes"),
]:
    if col in df.columns:
        print(f"\n{lbl}.{col}  (dtype={df[col].dtype}):")
        print(df[col].value_counts(dropna=False).head(15).to_string())

print("\n" + "=" * 78)
print("診斷結束。以上僅為證據輸出，未做任何過濾/排除/清理。")
print("=" * 78)


[5] schema 對映缺口清單

----------------------------------------------------------------------
[5a] 新資料 type 欄類別與數量（Kitten/Puppy 需併回 Cat/Dog）
----------------------------------------------------------------------
intakes.type：
type
Cat                   6208
Dog                   4932
Kitten                1644
Mammal                 661
Puppy                  447
Rabbit                  66
Guinea Pig              33
Bird (wildlife)         27
Reptile                 22
Fowl                    21
Bird                    20
Rodent                  16
Marsupial               13
Reptile (Wildlife)       9
Crab/fish                4
Ferret                   3
Sheep                    2
Goat                     2
Insect / Arachnid        1
Zoo                      1

intakes.type（Kitten->Cat, Puppy->Dog 併回後）：
type
Cat                   7852
Dog                   5379
Mammal                 661
Rabbit                  66
Guinea Pig              33
Bird (wildlife)         27
Reptile              